In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import librosa
import librosa.feature
import numpy as np
import librosa.display
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np
import os
import seaborn as sns
import shap

# define torch device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Feature extraction

In [ ]:
'''This is audio_processing.py for the speech analysis
'''

gender_dict = {
    'male': {
        'real': ['linus', 'ryan'],
        'fake': ['linus-to-musk', 'taylor-to-linus', 'linus-to-ryan']
    },
    'female':
    {
        'real': ['taylor', 'margot'],
        'fake': ['taylor-to-margot', 'linus-to-taylor', 'linus-to-margot']
    }
}

real_folder_path = '/content/drive/My Drive/LING_199/REAL_audio'
fake_folder_path = '/content/drive/My Drive/LING_199/FAKE_audio'

def extract_features(real_folder_path, fake_folder_path, gender_dict, gender='both'):
    """
    Extracts audio features from real and fake audio files based on gender specification.

    Parameters:
        real_folder_path (str): path to real audio folders.
        fake_folder_path (str): path to fake audio files.
        gender_dict (dict): Dictionary defining real and fake audio file names by gender.
        gender (str): 'male', 'female', or 'both'.

    Returns:
        DataFrame: Extracted features with labels (0 for real, 1 for fake).

    Function Outline:
    1. Subset audio files based on gender
    2. Open each audio file
        a. Split audio into 1 second frames
        b. Extract features from each frame
        c. Append features to a list
    3. Convert list to dataframe
    """

    # define helper function for processing each file
    def process_file(file_path, label):
        """Loads a file, extracts features from that file, and generates a label
           given a input for label
        """
        try:
            y, sr = librosa.load(file_path, sr=None)
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
            return []

        samples_per_second = sr
        frames = [y[i:i + samples_per_second] for i in range(0, len(y), samples_per_second)]
        features = []

        for frame in frames:
            if len(frame) < samples_per_second:
                continue
            # extract relevant features
            chromagram = librosa.feature.chroma_stft(y=frame, sr=sr).mean()
            rms = librosa.feature.rms(y=frame).mean()
            spectral_centroid = librosa.feature.spectral_centroid(y=frame, sr=sr).mean()
            spectral_bandwidth = librosa.feature.spectral_bandwidth(y=frame, sr=sr).mean()
            rolloff = librosa.feature.spectral_rolloff(y=frame, sr=sr).mean()
            zero_crossing_rate = librosa.feature.zero_crossing_rate(frame).mean()
            mfcc = librosa.feature.mfcc(y=frame, sr=sr, n_mfcc=20).mean(axis=1)

            frame_features = {
                'chroma_stft': chromagram,
                'rms': rms,
                'spectral_centroid': spectral_centroid,
                'spectral_bandwidth': spectral_bandwidth,
                'rolloff': rolloff,
                'zero_crossing_rate': zero_crossing_rate,
                **{f'mfcc{i+1}': mfcc[i] for i in range(20)},
                'LABEL': label
            }
            features.append(frame_features)

        return features

    # Select files based on gender
    if gender.lower() == 'male':
        real_files = gender_dict['male']['real']
        fake_files = gender_dict['male']['fake']
        real_path = os.path.join(real_folder_path, 'MALE')
    elif gender.lower() == 'female':
        real_files = gender_dict['female']['real']
        fake_files = gender_dict['female']['fake']
        real_path = os.path.join(real_folder_path, 'FEMALE')
    else:
        print('Must input male or female')
        return

    real_files = [f + '-original.wav' for f in real_files]
    fake_files = [f + '.wav' for f in fake_files]

    # Feature extraction

    # initiate a list of features
    # each element is a dictionary of features for one second of audio
    all_features = []

    # run process file for all real files, append features to list
    for file in real_files:
        path = os.path.join(real_path, file)
        print(f"Processing real: {path}")
        all_features.extend(process_file(path, label=0))

    # run process file for all fake files, append features to list
    for file in fake_files:
        path = os.path.join(fake_folder_path, file)
        print(f"Processing fake: {path}")
        all_features.extend(process_file(path, label=1))

    # convert list of features to dataframe, return
    return pd.DataFrame(all_features)


# Simple NN

In [ ]:
# Load a dataset (balanced, female, or male)
df = pd.read_csv("/content/drive/My Drive/LING_199/BALANCED-dataset.csv")

# rename 'label' to 'LABEL' if necessary
if 'label' in df.columns and 'LABEL' not in df.columns:
    df = df.rename(columns={'label': 'LABEL'})

# rename 'FAKE' as 1 and 'REAL' as 0 if necessary
if 'FAKE' in df['LABEL'].unique() and 'REAL' in df['LABEL'].unique():
    df['LABEL'] = df['LABEL'].map({'FAKE': 1, 'REAL': 0})

# undersample the fake class randomly to ensure balanced data
fake_samples = df[df['LABEL'] == 1]
real_samples = df[df['LABEL'] == 0]
fake_samples_undersampled = fake_samples.sample(n=len(real_samples), random_state=42)
df = pd.concat([real_samples, fake_samples_undersampled])

# Separate features and labels
X = df.drop(columns=['LABEL']).values
y = df['LABEL'].values

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# PyTorch Dataset Class
class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create dataset objects
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

# Define a simple neural network
class SimpleNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(SimpleNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(inplace=False),
            nn.Linear(128, 64),
            nn.ReLU(inplace=False),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.net(x)


# Training parameters
num_epochs = 10
batch_size = 32
learning_rate = 0.001
num_classes = len(np.unique(y))  # Number of unique labels

# 10-fold cross-validation
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"Fold {fold + 1}")

    train_subset = Subset(train_dataset, train_idx)
    val_subset = Subset(train_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

    # Initialize model, loss function, and optimizer
    model = SimpleNN(input_size=X.shape[1], num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        # Validation loop
        model.eval()
        val_loss = 0.0
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                correct += (predicted == labels).sum().item()
                total += labels.size(0)

        accuracy = correct / total
        print(f"Epoch {epoch+1}: Train Loss: {running_loss/len(train_loader):.4f}, "
              f"Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {accuracy:.4f}")

# Final evaluation on the test set
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f"Test Set Accuracy: {correct / total:.4f}")

# Complex NN


In [ ]:
# Load a dataset (balanced, female, or male)
df = pd.read_csv("/content/drive/My Drive/LING_199/BALANCED-dataset.csv")

# rename 'label' to 'LABEL' if necessary
if 'label' in df.columns and 'LABEL' not in df.columns:
    df = df.rename(columns={'label': 'LABEL'})

# Convert LABEL column: 'REAL' -> 0, 'FAKE' -> 1 if necessary
if 'FAKE' in df['LABEL'].unique() and 'REAL' in df['LABEL'].unique():
    df['LABEL'] = df['LABEL'].map({'FAKE': 1, 'REAL': 0})

# Separate features and labels
X = df.drop(columns=['LABEL']).values
y = df['LABEL'].values

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# PyTorch Dataset
class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)  # Ensure shape compatibility

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create dataset objects
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

class ImprovedComplexNN(nn.Module):
    def __init__(self, input_size):
        super(ImprovedComplexNN, self).__init__()

        self.fc1 = nn.Linear(input_size, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.act1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)

        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.act2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.act3 = nn.ReLU()
        self.dropout3 = nn.Dropout(0.3)

        self.fc4 = nn.Linear(64, 32)
        self.bn4 = nn.BatchNorm1d(32)  # Added BatchNorm
        self.act4 = Swish()  # Swish activation function

        self.fc5 = nn.Linear(32, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.act1(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.bn2(x)
        x = self.act2(x)
        x = self.dropout2(x)

        x = self.fc3(x)
        x = self.bn3(x)
        x = self.act3(x)
        x = self.dropout3(x)

        x = self.fc4(x)
        x = self.bn4(x)
        x = self.act4(x)

        x = self.fc5(x)
        return torch.sigmoid(x)  # Apply Sigmoid at the output layer

# Training parameters
num_epochs = 10
batch_size = 32
learning_rate = 0.001

# 10-fold cross-validation
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"Fold {fold + 1}")

    train_subset = Subset(train_dataset, train_idx)
    val_subset = Subset(train_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

    # Initialize the ImprovedComplexNN model
    model = ImprovedComplexNN(input_size=X.shape[1]).to(device)
    criterion = nn.BCELoss()  # Binary Cross Entropy for binary classification
    optimizer = optim.Adam(model.parameters(), lr=learning_rate) # Adam optimizer

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        # Validation loop

        # set the model to evaluation mode
        model.eval()
        val_loss = 0.0
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                predicted = (outputs > 0.5).float()  # Convert probabilities to binary predictions
                correct += (predicted == labels).sum().item()
                total += labels.size(0)

        accuracy = correct / total
        print(f"Epoch {epoch+1}: Train Loss: {running_loss/len(train_loader):.4f}, "
              f"Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {accuracy:.4f}")

# Final evaluation on the test set
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        predicted = (outputs > 0.5).float()
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f"Final Test Accuracy: {correct / total:.4f}")

# XGBoost Model


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import xgboost as xgb

# load a dataset (balanced, male, or female)
df = pd.read_csv("/content/drive/My Drive/LING_199/BALANCED-dataset.csv")

# rename 'label' to 'LABEL' if needed
if 'label' in df.columns and 'LABEL' not in df.columns:
    df.rename(columns={'label': 'LABEL'}, inplace=True)

# convert all 'REAL' to 0 and 'FAKE' to 1 in the 'LABEL' column if REAL and FAKE are in the 'LABEL' column
if 'REAL' in df['LABEL'].values and 'FAKE' in df['LABEL'].values:
    df['LABEL'] = df['LABEL'].replace({'REAL': 0, 'FAKE': 1})

# 1. Prepare the data
X = df.drop('LABEL', axis=1)  # Features (all columns except 'LABEL')
y = df['LABEL']               # Target variable

# 2. Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,    # 20% for testing
    random_state=42,
    stratify=y        # Maintain class distribution
)

# Create DMatrix for XGBoost (optimized data structure)
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# Set XGBoost parameters
params = {
    'objective': 'binary:logistic',  # Binary classification
    'eval_metric': 'logloss',       # Logarithmic loss
    'eta': 0.1,                     # Learning rate
    'max_depth': 6,                 # Maximum tree depth
    'subsample': 0.8,               # Subsample ratio
    'colsample_bytree': 0.8,        # Feature subsample ratio
    'seed': 42,
    'early_stopping_rounds': 10
}

# Train the model
num_rounds = 100
eval_list = [(dtrain, 'train'), (dtest, 'eval')]

model = xgb.train(
    params,
    dtrain,
    num_rounds,
    evals=eval_list,
    verbose_eval=10
)